# Introducción

## Librería

In [ ]:
import os  # Interacción con el sistema operativo: manejar rutas, archivos y variables de entorno
from pathlib import Path  # Manejo seguro y portátil de rutas de archivos y carpetas
import warnings  # Control de advertencias en tiempo de ejecución
import math  # Funciones matemáticas estándar (raíz cuadrada, trigonometría, logaritmos, etc.)
import random  # Generación de números aleatorios y control de semilla
import tensorflow as tf

import keras  # Framework de alto nivel para construir y entrenar redes neuronales
from keras import layers  # Tipos de capas en redes neuronales (Dense, Conv2D, LSTM...)

import numpy as np  # Operaciones numéricas eficientes con arrays y matrices
import pandas as pd  # Manipulación y análisis de datos en estructuras DataFrame
import matplotlib.pyplot as plt  # Visualización de datos (gráficos, histogramas, curvas)
from keras.applications import EfficientNetB0 # Importa la versión B0 de EfficientNet, un modelo preentrenado en ImageNet
import tensorflow_datasets as tfds # Importa TensorFlow Datasets, para cargar datasets predefinidos de manera fácil

warnings.filterwarnings("ignore")  # Ignorar todas las advertencias para evitar ruido en la salida
# os.environ["KERAS_BACKEND"] = "jax"  # Configuración opcional: usar JAX como backend de Keras
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Mostrar solo advertencias y errores de TensorFlow (oculta info y logs)

from tensorflow import keras  # Usar Keras integrado en TensorFlow 2.x
from tensorflow.keras import layers  # Acceso a capas de Keras dentro de TensorFlow
from tensorflow import data as tf_data  # Submódulo para manejo eficiente de datasets y pipelines
import zipfile  # Módulo de Python para trabajar con archivos ZIP


seed = 13  # Valor fijo para reproducibilidad de resultados
np.random.seed(seed)  # Semilla para operaciones aleatorias de NumPy
random.seed(seed)  # Semilla para el módulo random de Python
keras.utils.set_random_seed(seed)  # Semilla para reproducibilidad en TensorFlow/Keras

from IPython.core.magic import register_cell_magic  # Permite crear "magias" personalizadas de celda en Jupyter/Colab

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Silencia avisos de compilación y advertencias menores

import tensorflow as tf
# Opcional: Forzar a que solo se muestren errores críticos
tf.get_logger().setLevel('ERROR')

In [ ]:
# Configura el tamaño de fuente general por defecto en los gráficos
plt.rc('font', size=14)

# Configura el tamaño de las etiquetas de los ejes (x, y) y el tamaño del título del gráfico
plt.rc('axes', labelsize=14, titlesize=14)

# Configura el tamaño de la fuente de la leyenda del gráfico
plt.rc('legend', fontsize=14)

# Configura el tamaño de las etiquetas en el eje x (números o categorías)
plt.rc('xtick', labelsize=10)

# Configura el tamaño de las etiquetas en el eje y (números o categorías)
plt.rc('ytick', labelsize=10)

## Funciones

In [ ]:
# Crea la ruta donde se guardarán las imágenes
IMAGES_PATH = Path() / "images" / "CNN-IPN_mod6"

# Crea la carpeta si no existe
# parents=True permite crear también carpetas intermedias
# exist_ok=True evita error si la carpeta ya existe
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

# Función para guardar figuras generadas con matplotlib
def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):

    # Construye la ruta completa del archivo usando el nombre de la figura
    # Ejemplo: images/ann/grafica1.png
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"

    # Ajusta automáticamente los espacios del gráfico para evitar que se encimen etiquetas o títulos
    if tight_layout:
        plt.tight_layout()

    # Guarda la figura en la ruta especificada
    # format = tipo de archivo (png, jpg, pdf, etc.)
    # dpi = resolución de la imagen (300 es calidad alta para artículos o reportes)
    plt.savefig(path, format=fig_extension, dpi=resolution)


def mostrar_imagen_grises(imagen, cmap="binary"):
    """
    Muestra una imagen en escala de grises.

    Parámetros:
    imagen : array-like
        Imagen a mostrar.
    cmap : str, opcional
        Mapa de color (por defecto 'binary').

    Nota:
    - Valores pequeños → blanco
    - Valores grandes → negro
    """
    plt.imshow(imagen, cmap=cmap)
    plt.axis('off')  # Oculta los ejes
    plt.show()       # Muestra la imagen

def mostrar_guardar_imagenes(n_rows, n_cols, x_train, y_train, nombre_guardar, titulo):
    """
    Genera una figura con una cuadrícula de imágenes, coloca títulos a cada imagen,
    agrega un título general y guarda la figura en un archivo.

    Parámetros:
        n_rows (int): Número de filas en la cuadrícula.
        n_cols (int): Número de columnas en la cuadrícula.
        x_train (array-like): Conjunto de imágenes a mostrar.
        y_train (array-like): Etiquetas de las imágenes.
        nombre_guardar (str): Nombre del archivo para guardar la figura.
        titulo (str): Título general de la figura.
    """

    plt.figure(figsize=(n_cols * 1.2, n_rows * 1.2))

    # Título general
    plt.suptitle(titulo, fontsize=16)

    for row in range(n_rows):
        for col in range(n_cols):
            index = n_cols * row + col
            if index >= len(x_train):
                break  # Evita errores si x_train tiene menos imágenes

            plt.subplot(n_rows, n_cols, index + 1)
            plt.imshow(x_train[index], cmap="binary", interpolation="nearest")
            plt.axis('off')
            plt.title(y_train[index], fontsize=8)

    plt.subplots_adjust(wspace=0.2, hspace=0.5, top=0.88)  # Ajusta el espacio para el título
    save_fig(nombre_guardar)  # Guarda la figura
    plt.show()  # Muestra la figura



def plot_learning_curves_clasificacion(history, nombre_imagen):
    # Convertir a DataFrame
    df = pd.DataFrame(history.history)

    # Crear figura
    fig, ax = plt.subplots(figsize=(8, 5))

    # Graficar pérdida
    if 'loss' in df and 'val_loss' in df:
        ax.plot(df.index + 1, df['loss'], "r--", label="Train Loss")
        ax.plot(df.index + 1, df['val_loss'], "b--.", label="Val Loss")

    # Graficar accuracy (clasificación)
    if 'accuracy' in df and 'val_accuracy' in df:
        ax.plot(df.index + 1, df['accuracy'], "r-", label="Train Accuracy")
        ax.plot(df.index + 1, df['val_accuracy'], "b-*", label="Val Accuracy")

    # Configuración de ejes
    ax.set_xlim(1, len(df))
    ax.set_ylim(0, 1)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Metric Value")
    ax.grid(True)

    # Leyenda fuera del gráfico
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # Guardar y mostrar
    plt.tight_layout()
    plt.savefig(nombre_imagen)
    plt.show()



def mostrar_imagenes_RBG(dataset, class_names, nombre_guardar, titulo='', num_imagenes=10, columnas=5):

    """
    Muestra imágenes de un dataset de TensorFlow en una cuadrícula ajustable y guarda la figura.
    """

    filas = math.ceil(num_imagenes / columnas)  # Calcula número de filas necesarias
    plt.figure(figsize=(3*columnas, 3*filas))  # Crea la figura con tamaño proporcional

    titulo = f"Visualización de {num_imagenes} imágenes de ejemplo {titulo}"  # Construye título principal
    plt.suptitle(titulo, fontsize=16)  # Coloca título en la figura

    cont = 0  # Contador de subplots

    for images, labels in dataset.take(1):  # Itera sobre la primera tanda de imágenes
        for i in range(min(num_imagenes, len(images))):  # Limita al número de imágenes deseadas
            ax = plt.subplot(filas, columnas, cont + 1)  # Crea subplot en cuadrícula
            plt.imshow(np.array(images[i]).astype("uint8"))  # Muestra la imagen en uint8
            label_idx = int(labels[i])  # Obtiene índice de clase
            plt.title(f"{class_names[label_idx]} - ({label_idx})")  # Título de cada imagen
            plt.axis("off")  # Quita ejes
            cont += 1  # Incrementa contador de subplots

    plt.subplots_adjust(wspace=0.2, hspace=0.5, top=0.88)  # Ajusta espacios entre subplots y título
    save_fig(f"{nombre_guardar}.png")  # Guarda la figura en archivo PNG
    plt.show()  # Muestra la figura en pantalla


def mostrar_imagenes_RBG2(dataset, class_names, nombre_guardar=None, titulo='', num_imagenes=10, columnas=5):
    """
    Muestra imágenes de un dataset de TensorFlow en un grid ajustable y opcionalmente guarda la figura.
    Funciona tanto para datasets en batch como para datasets sin batch.
    """

    filas = math.ceil(num_imagenes / columnas)  # Calcula el número de filas necesarias para el grid
    plt.figure(figsize=(3*columnas, 3*filas))  # Ajusta el tamaño de la figura según columnas y filas

    if titulo:
        titulo = f"Visualización de {num_imagenes} imágenes de ejemplo {titulo}"  # Construye título principal
        plt.suptitle(titulo, fontsize=16)  # Coloca título en la figura

    cont = 0  # Contador de imágenes mostradas

    for element in dataset.take(num_imagenes):  # Itera sobre el dataset, limitado a num_imagenes
        if isinstance(element, tuple) and len(element) == 2:  # Verifica que el elemento sea (images, labels)
            images, labels = element  # Separa imágenes y etiquetas

            if len(images.shape) > 3:  # Caso batch: varias imágenes juntas (batch, H, W, C)
                for i in range(len(images)):  # Itera sobre cada imagen en el batch
                    if cont >= num_imagenes:  # Rompe si se alcanza el límite
                        break
                    plt.subplot(filas, columnas, cont + 1)  # Crea un subplot en la cuadrícula
                    plt.imshow(np.array(images[i]).astype("uint8"))  # Muestra la imagen convertida a uint8
                    plt.title(f"{class_names[int(labels[i])]} - ({int(labels[i])})")  # Título con nombre y índice de clase
                    plt.axis("off")  # Quita los ejes
                    cont += 1  # Incrementa contador
            else:  # Caso imagen individual: (H, W, C)
                plt.subplot(filas, columnas, cont + 1)  # Crea subplot
                plt.imshow(np.array(images).astype("uint8"))  # Muestra imagen
                plt.title(f"{class_names[int(labels)]} - ({int(labels)})")  # Título con nombre e índice de clase
                plt.axis("off")  # Quita los ejes
                cont += 1  # Incrementa contador

    plt.subplots_adjust(wspace=0.3, hspace=0.5, top=0.88)  # Ajusta espacio entre subplots y margen superior

    if nombre_guardar:  # Si se proporciona un nombre de archivo
        plt.savefig(f"{nombre_guardar}.png")  # Guarda la figura como PNG

    plt.show()  # Muestra la figura en pantalla

def mostrar_imagenes_RGB3(dataset, class_names, nombre_guardar=None, titulo='', num_imagenes=10, columnas=5):
    import math
    filas = math.ceil(num_imagenes / columnas)
    plt.figure(figsize=(3*columnas, 3*filas))

    if titulo:
        plt.suptitle(f"Visualización: {titulo}", fontsize=16)

    cont = 0
    for element in dataset.take(num_imagenes):
        images, labels = element if (isinstance(element, tuple) and len(element) == 2) else (element, None)

        # Normalizar a lista para iterar igual batch o single
        if len(images.shape) > 3:
            curr_imgs, curr_lbls = images, labels
        else:
            curr_imgs, curr_lbls = [images], [labels]

        for i in range(len(curr_imgs)):
            if cont >= num_imagenes: break

            plt.subplot(filas, columnas, cont + 1)
            img = curr_imgs[i].numpy() if hasattr(curr_imgs[i], 'numpy') else curr_imgs[i]

            # Mostrar según tipo de dato (float 0-1 o int 0-255)
            if img.dtype.kind == 'f':
                plt.imshow(np.clip(img, 0, 1))
            else:
                plt.imshow(img.astype("uint8"))

            if curr_lbls is not None:
                idx = int(curr_lbls[i])
                plt.title(f"{class_names[idx]}\n({idx})", fontsize=10)

            plt.axis("off")
            cont += 1

    plt.subplots_adjust(wspace=0.3, hspace=0.6, top=0.85)

    if nombre_guardar:
        # Usamos plt.savefig directamente para evitar el error de save_fig()
        save_fig(f"{nombre_guardar}.png")
        print(f"Imagen guardada como: {nombre_guardar}.png")

    plt.show()

def resumen_dataset_imagekeras(dataset, nombre):
    # Creamos un arreglo de ceros con tamaño igual al número de clases
    # Aquí iremos acumulando cuántas imágenes hay por clase
    counts = np.zeros(len(dataset.class_names), dtype=int)

    # Recorremos el dataset batch por batch
    # _ representa las imágenes (no las usamos aquí)
    # labels son las etiquetas de cada imagen en el batch
    for _, labels in dataset:
        # Recorremos cada etiqueta dentro del batch
        for label in labels:
            # label.numpy() convierte el tensor a número (ej: 0, 1, 2...)
            # Sumamos 1 al contador de esa clase
            counts[label.numpy()] += 1

    # Calculamos el total de imágenes sumando todos los conteos
    total = counts.sum()

    # Imprimimos el nombre del dataset (Train o Validation)
    print(f"\n        {nombre}        ")
    print(f"Total imágenes: {total}\n")

    # Encabezados de la tabla
    print(f"{'Clase':<15}{'Cantidad':<10}{'Proporción'}")
    print("-"*35)

    # Recorremos cada clase para mostrar sus datos
    for i, class_name in enumerate(dataset.class_names):
        # Calculamos la proporción de esa clase respecto al total
        propor = counts[i] / total

        # Mostramos: nombre de la clase, cantidad y proporción
        print(f"{class_name:<15}{counts[i]:<10}{propor:.3f}")


def resumen_dataset_imagekeras2(dataset, class_names, nombre):
    """
    Resume el dataset mostrando cantidad de imágenes por clase y proporción.
    Funciona tanto para datasets con batch como sin batch.

    Args:
        dataset: tf.data.Dataset que devuelve (imagen, label)
        class_names: lista de nombres de clases
        nombre: nombre descriptivo del dataset (ej. "Train" o "Validation")
    """

    # Inicializamos contador por clase
    counts = np.zeros(len(class_names), dtype=int)

    for element in dataset:  # Itera sobre todos los elementos
        if isinstance(element, tuple) and len(element) == 2:
            images, labels = element

            # Caso batch (varias etiquetas)
            if len(labels.shape) > 0:
                for label in labels:
                    counts[int(label.numpy())] += 1
            else:  # Caso etiqueta individual
                counts[int(labels.numpy())] += 1

    total = counts.sum()  # Total de imágenes

    # Mostramos resumen
    print(f"\n        {nombre}        ")
    print(f"Total imágenes: {total}\n")

    print(f"{'Clase':<15}{'Cantidad':<10}{'Proporción'}")
    print("-"*35)

    for i, class_name in enumerate(class_names):
        propor = counts[i] / total
        print(f"{class_name:<15}{counts[i]:<10}{propor:.3f}")

# Función que aplica el aumento de datos a un batch de imágenes
def data_augmentation(images):
    # Recorre cada capa de aumento definida
    for layer in data_augmentation_layers:
        # Aplica la transformación actual a las imágenes
        images = layer(images)

    # Devuelve las imágenes transformadas
    return images






def mostrar_imagenes_aumentadas_imageskeras(dataset, class_names, nombre_guardar, titulo='',
                                num_imagenes=3, num_aumentos=4, columnas=4):
    """
    Muestra imágenes aumentadas (data augmentation) de un dataset en una cuadrícula
    y guarda la figura.

    - num_imagenes: número de imágenes distintas a usar
    - num_aumentos: número de transformaciones por imagen
    """

    filas = num_imagenes  # una fila por imagen base
    plt.figure(figsize=(3*columnas, 3*filas))  # tamaño proporcional

    titulo = f"Imágenes aumentadas ({num_imagenes}x{num_aumentos}) {titulo}"
    plt.suptitle(titulo, fontsize=16)

    cont = 0  # contador de subplots

    for images, labels in dataset.take(1):  # tomamos un batch

        for j in range(min(num_imagenes, len(images))):  # imágenes distintas

            img = images[j:j+1]  # batch de tamaño 1
            label = int(labels[j])  # etiqueta

            for i in range(num_aumentos):  # aumentos por imagen

                augmented_img = data_augmentation(img)  # aplicar augmentation

                ax = plt.subplot(filas, columnas, cont + 1)

                plt.imshow(np.array(augmented_img[0]).astype("uint8"))

                # título con clase y número de aumento
                plt.title(f"{class_names[label]} ({label}) - aug {i+1}")

                plt.axis("off")
                cont += 1

    plt.subplots_adjust(wspace=0.2, hspace=0.5, top=0.88)
    save_fig(f"{nombre_guardar}.png")
    plt.show()



def mostrar_imagenes_aumentadas_imageskeras2(dataset, class_names, nombre_guardar, titulo='',
                                             num_imagenes=1, num_aumentos=9, columnas=3):
    """
    Muestra imágenes aumentadas de un dataset de TensorFlow/Keras y guarda la figura.

    Parámetros:
    - dataset: conjunto de datos de TensorFlow (images, labels)
    - class_names: lista de nombres de clases
    - nombre_guardar: ruta/nombre del archivo para guardar la figura
    - titulo: título opcional de la figura
    - num_imagenes: número de imágenes originales a mostrar
    - num_aumentos: número de aumentos por imagen
    - columnas: número de columnas en la figura
    """

    total_imgs = num_imagenes * num_aumentos  # total de imágenes a mostrar
    filas = math.ceil(total_imgs / columnas)  # filas necesarias

    plt.figure(figsize=(4 * columnas, 3.5 * filas))  # tamaño de la figura
    plt.suptitle(f"Imágenes aumentadas ({num_imagenes}x{num_aumentos}) {titulo}", fontsize=16)

    cont = 0  # contador de subplots
    imagenes_mostradas = 0  # contador de imágenes originales procesadas

    # iterar sobre el dataset hasta obtener num_imagenes
    for images, labels in dataset:
        # asegurar batch
        if len(images.shape) == 3:
            images = np.expand_dims(images, axis=0)
            labels = np.array([labels])

        for j in range(len(images)):
            if imagenes_mostradas >= num_imagenes:
                break  # ya alcanzamos el número de imágenes deseadas

            image = images[j]
            label = int(labels[j])
            image_np = image.numpy() if hasattr(image, "numpy") else image

            # generar num_aumentos por cada imagen
            for i in range(num_aumentos):
                ax = plt.subplot(filas, columnas, cont + 1)

                aug_img = data_augmentation(np.expand_dims(image_np, axis=0))
                aug_img = np.array(aug_img)

                plt.imshow(aug_img[0].astype("uint8"))
                plt.title(f"{class_names[label]} ({label}) - aug {i+1}", pad=12, fontsize=9)
                plt.axis("off")
                cont += 1

            imagenes_mostradas += 1

        if imagenes_mostradas >= num_imagenes:
            break

    plt.subplots_adjust(wspace=0.3, hspace=0.9, top=0.88)
    save_fig(f"{nombre_guardar}.png")
    plt.show()


def mostrar_imagenes_aumentadas_imageskeras3(dataset, class_names, nombre_guardar, titulo='',
                                           num_imagenes=1, num_aumentos=9, columnas=3):
    """
    Muestra imágenes aumentadas recorriendo la LISTA de capas de data_augmentation.
    Maneja automáticamente imágenes normalizadas (0-1).
    """

    total_imgs = num_imagenes * num_aumentos
    filas = math.ceil(total_imgs / columnas)

    plt.figure(figsize=(4 * columnas, 3.5 * filas))
    plt.suptitle(f"Imágenes aumentadas ({num_imagenes}x{num_aumentos}) {titulo}", fontsize=16)

    cont = 0
    imagenes_mostradas = 0

    for images, labels in dataset:
        # Aseguramos que tratamos los datos como tensores para las capas de Keras
        if len(images.shape) == 3:
            images = tf.expand_dims(images, axis=0)
            labels = np.array([labels])

        for j in range(len(images)):
            if imagenes_mostradas >= num_imagenes:
                break

            # Extraemos la imagen original del batch
            original_image = images[j]
            label = int(labels[j])

            # Generar los aumentos solicitados para esta imagen específica
            for i in range(num_aumentos):
                plt.subplot(filas, columnas, cont + 1)

                # --- PROCESO DE AUMENTO CAPA POR CAPA ---
                # Empezamos con la imagen original (añadiendo dimensión de batch)
                aug_img = tf.expand_dims(original_image, axis=0)

                # Iteramos sobre la LISTA de capas (data_augmentation_layers)
                for layer in data_augmentation_layers:
                    aug_img = layer(aug_img, training=True)

                # Quitamos la dimensión de batch [1, H, W, C] -> [H, W, C]
                aug_img = tf.squeeze(aug_img, axis=0).numpy()

                # --- VISUALIZACIÓN INTELIGENTE ---
                # Si los valores son float (0.0 a 1.0), usamos clip para evitar errores
                if aug_img.dtype.kind == 'f':
                    plt.imshow(np.clip(aug_img, 0, 1))
                else:
                    plt.imshow(aug_img.astype("uint8"))

                plt.title(f"{class_names[label]}\n(aug {i+1})", fontsize=9)
                plt.axis("off")
                cont += 1

            imagenes_mostradas += 1

        if imagenes_mostradas >= num_imagenes:
            break

    plt.subplots_adjust(wspace=0.3, hspace=0.6, top=0.9)

    # Guardado seguro usando matplotlib directamente
    if nombre_guardar:
        plt.savefig(f"{nombre_guardar}.png", bbox_inches='tight')
        print(f"Figura guardada exitosamente como: {nombre_guardar}.png")

    plt.show()

# ================================
# One-hot / categorical encoding
# ================================

# Función de preprocesamiento para datos de entrenamiento
def input_preprocess_train(image, label, num_classes):
    image = data_augmentation(image)     # Aplica aumentos de datos a la imagen (rotaciones, flips, cambios de brillo, etc.)
    label = tf.one_hot(label, num_classes)     # Convierte la etiqueta en codificación one-hot según num_classes
    return image, label     # Retorna la imagen y etiqueta preprocesadas


# Función de preprocesamiento para datos de prueba (sin augmentación)
def input_preprocess_test(image, label, num_classes):
    # Solo codifica la etiqueta a one-hot
    label = tf.one_hot(label, num_classes)
    return image, label

# VGG16

In [ ]:
IMG_SIZE = 224 # Define el tamaño de la imagen de entrada. EfficientNetB0 espera imágenes de 224x224 píxeles
BATCH_SIZE = 64 # Define el tamaño del batch, es decir, cuántas imágenes se procesan a la vez durante el entrenamiento

In [ ]:
dataset_name = "stanford_dogs"
# Cargamos el dataset "Stanford Dogs" usando TensorFlow Datasets
(ds_train, ds_test), ds_info = tfds.load(
    "stanford_dogs",         # Nombre del dataset a cargar
    split=["train", "test"], # Separar en conjunto de entrenamiento y prueba
    with_info=True,          # También devuelve información del dataset (ds_info)
    as_supervised=True,      # Devuelve los datos como pares (imagen, etiqueta)
    data_dir="./tfds_data"   # Carpeta donde se descargará o leerá el dataset
)

Información de la data

In [ ]:
train_size = ds_info.splits['train'].num_examples
test_size = ds_info.splits['test'].num_examples

print(f"Número de imágenes de entrenamiento: {train_size}")
print(f"Número de imágenes de prueba: {test_size}")

In [ ]:
class_names = ds_info.features['label'].names
print(f"""
Clases (las primeras 5): {class_names[:5]}
Número de clases: {len(class_names)}""")
class_names = [name.split("-")[1] for name in class_names] # Aplicamos split("-") para quedarnos solo con la parte después del guion
print(f"""
Clases (las primeras 5) sin split: {class_names[:5]}
 """)

In [ ]:
resumen_dataset_imagekeras2(ds_train,class_names, "Conjunto de entrenamiento")
resumen_dataset_imagekeras2(ds_test,class_names, "Conjunto de prueba")

Obtengo el tamaño de las primeras $10$ imágenes

In [ ]:
print("Dimensiones y rangos de valores (Máx, Mín) de las primeras 10 imágenes:")
for i, (image, label) in enumerate(ds_train.take(10)):
    # Calculamos los valores máximo y mínimo del tensor
    max_val = tf.reduce_max(image).numpy()
    min_val = tf.reduce_min(image).numpy()

    print(f"Imagen {i+1}: {image.shape} | Máximo: {max_val} | Mínimo: {min_val}")

Revisar la información del dataset

In [ ]:
print(f"Tipo de etiqueta en ds_info: {ds_info.features['label']}")

# Opción 2: Ver un ejemplo real
for _, label in ds_train.take(1):
    print(f"Valor de la etiqueta: {label.numpy()}")
    print(f"Forma del tensor de la etiqueta: {label.shape}")

In [ ]:
size = (IMG_SIZE, IMG_SIZE)
size

## Visualización

### Entrenamiento - Sin efectos

In [ ]:
mostrar_imagenes_RGB3(ds_train, class_names, num_imagenes=20, nombre_guardar= 'Visualizacion_ejemplo_stanford_dogs_entrenamiento' ,
                     titulo='stanford dogs(entrenamiento)')  # Muestra 12 imágenes, 5 columnas

### Prueba - Sin efectos

In [ ]:
mostrar_imagenes_RGB3(ds_test, class_names, num_imagenes=20, nombre_guardar= 'Visualizacion_ejemplo_stanford_dogs_prueba' ,
                     titulo='stanford dogs(prueba)')  # Muestra 12 imágenes, 5 columnas

## Preprocesamiento

### Data augmentation

In [ ]:
# Lista de capas de data augmentation (aumentos artificiales de imágenes)
img_augmentation_layers = [
    # Rota la imagen aleatoriamente dentro de un rango factor=0.15 (~±54°)
    layers.RandomRotation(factor=0.15),

    # Desplaza la imagen horizontal (10%) y verticalmente (10%)
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),

    # Voltea la imagen aleatoriamente de forma horizontal
    layers.RandomFlip(mode="horizontal"),

    # Ajusta el contraste de la imagen aleatoriamente
    layers.RandomContrast(factor=0.1),

    # Zoom aleatorio para que el modelo aprenda a ver perros de cerca y lejos
    layers.RandomZoom(height_factor=0.1, width_factor=0.1),

    #  Brillo aleatorio para simular diferentes condiciones de iluminación
    layers.RandomBrightness(factor=0.1)
]

In [ ]:
# Creamos un modelo secuencial solo para el aumento de datos
data_augmentation_model = tf.keras.Sequential(img_augmentation_layers)

In [ ]:
def preprocess_imageVGG16(image, label):
    """
    Paso 1: Redimensión y normalización matemática para VGG16.
    """
    # Redimensionar al tamaño de entrada de la red (224, 224)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    # Preprocesamiento oficial: resta la media de ImageNet y cambia canales a BGR
    image = tf.keras.applications.vgg16.preprocess_input(image)
    return image, label
def apply_augmentation(images, labels):
    """
    Paso 2: Aplicar las capas de aumento al batch de imágenes.
    """
    images = data_augmentation_model(images, training=True)
    return images, labels

def mostrar_imagenes_VGG16(dataset, class_names, nombre_guardar=None, titulo='', num_imagenes=10, columnas=5):
    import math
    import numpy as np
    import matplotlib.pyplot as plt

    filas = math.ceil(num_imagenes / columnas)
    plt.figure(figsize=(3*columnas, 3*filas))

    if titulo:
        plt.suptitle(f"Visualización: {titulo}", fontsize=16)

    cont = 0
    # Iteramos sobre el dataset (que ya viene en batches de 32)
    for images, labels in dataset.take(math.ceil(num_imagenes/BATCH_SIZE)):

        for i in range(len(images)):
            if cont >= num_imagenes: break

            plt.subplot(filas, columnas, cont + 1)

            # Convertir a numpy
            img = images[i].numpy() if hasattr(images[i], 'numpy') else images[i]

            # --- AJUSTE PARA VGG16 ---
            # Las imágenes procesadas por VGG16 no están en rango [0,1] ni [0,255].
            # Para visualizarlas, las re-escalamos al rango [0, 1] de forma lineal:
            img_min = img.min()
            img_max = img.max()
            img_visual = (img - img_min) / (img_max - img_min)

            plt.imshow(img_visual)

            # Mostrar etiqueta
            if labels is not None:
                idx = int(labels[i])
                # Evitar error si el índice está fuera de rango de class_names
                label_name = class_names[idx] if idx < len(class_names) else f"Clase {idx}"
                plt.title(f"{label_name}\n({idx})", fontsize=10)

            plt.axis("off")
            cont += 1

    plt.subplots_adjust(wspace=0.3, hspace=0.6, top=0.85)

    if nombre_guardar:
        # Nota: Asegúrate de tener definida la función save_fig o usa plt.savefig
        plt.savefig(f"{nombre_guardar}.png")
        print(f"Imagen guardada como: {nombre_guardar}.png")

    plt.show()


In [ ]:
num_classes = len(class_names)
num_classes

In [ ]:
def to_one_hot(image, label):
    """ Convierte el índice entero a un vector One-Hot de tamaño num_classes """
    # num_classes debe ser 120 para Stanford Dogs
    label = tf.one_hot(label, num_classes)
    return image, label

# Aplicar al conjunto de ENTRENAMIENTO
ds_train_process = (
    # 1. Cargamos el dataset original de entrenamiento
    ds_train

    # 2. Mezclamos los datos (Shuffle):
    # Usamos un buffer de 1000 imágenes para desordenarlas.
    # Esto es vital para que el modelo no aprenda el orden de las etiquetas (recursión).
    .shuffle(1000)

    # 3. Mapeo de preprocesamiento (Resize + VGG16 Preprocess):
    # Redimensiona las imágenes a 224x224. 'AUTOTUNE' permite que TensorFlow
    # use todos los núcleos de tu CPU disponibles para procesar en paralelo.
    .map(preprocess_imageVGG16, num_parallel_calls=tf.data.AUTOTUNE)

    # 4. Agrupamiento en lotes (Batching):
    # Agrupamos las imágenes en grupos de (BATCH_SIZE). El modelo se actualiza tras cada lote.
    .batch(BATCH_SIZE)

    # 5. Aplicación de Aumento de Datos (Data Augmentation):
    # Aquí es donde ocurre la magia: cada vez que el modelo pida un lote, estas
    # capas aplicarán rotación, zoom o flips ALEATORIOS. Así, el modelo nunca ve la misma imagen exacta.
    .map(apply_augmentation, num_parallel_calls=tf.data.AUTOTUNE)

    # 6. Prefetch (Carga anticipada):
    # Mientras la GPU está entrenando con un lote, la CPU prepara el siguiente.
    # Esto evita cuellos de botella y acelera drásticamente el entrenamiento.
    .prefetch(tf.data.AUTOTUNE)
)

# Aplicar al conjunto de PRUEBA
ds_test_process = (
    # 1. Cargamos el dataset original de prueba
    ds_test

    # 2. Aplicamos el preprocesamiento básico (Resize + VGG16 Preprocess):
    # Usamos la misma función 'preprocess_image' que usamos en entrenamiento
    # para que las imágenes tengan el mismo formato (224x224 y normalización VGG).
    .map(preprocess_imageVGG16, num_parallel_calls=tf.data.AUTOTUNE)

    # 3. Agrupamiento en lotes (Batching):
    # drop_remainder=True descarta el último lote si no completa los 32 elementos,
    # esto ayuda a mantener formas constantes en tensores durante la evaluación.
    .batch(batch_size=BATCH_SIZE, drop_remainder=True)

    # 4. Prefetch (Carga anticipada):
    # Preparamos los lotes en memoria para que la evaluación sea rápida.
    .prefetch(tf.data.AUTOTUNE)
)

### Visualización

In [ ]:
mostrar_imagenes_VGG16(
    dataset=ds_train_process,
    class_names=class_names,
    titulo='Imágenes de Entrenamiento (VGG16 + Augmentation)',
    num_imagenes=10
)

Observamos que hay aleatoriedad

In [ ]:
mostrar_imagenes_VGG16(
    dataset=ds_train_process,
    class_names=class_names,
    titulo='Imágenes de Entrenamiento (VGG16 + Augmentation)',
    num_imagenes=10
)

En la visualización de los datos de prueba, no se hace aleatoriedad y tampoco image augmentation

In [ ]:
mostrar_imagenes_VGG16(
    dataset=ds_test_process,
    class_names=class_names,
    titulo='Imágenes de Prueba (VGG16 + Augmentation)',
    num_imagenes=10
)

In [ ]:
# ==========================================
# INSPECCIÓN DE ETIQUETAS (TRAIN VS TEST)
# ==========================================

def inspeccionar_etiquetas(dataset, nombre_ds, n=5):
    print(f"\n--- Inspeccionando primeras {n} etiquetas de: {nombre_ds} ---")

    # .unbatch() nos permite ver los elementos individuales
    # sin importar si el dataset ya tiene aplicado el .batch()
    for i, (imagen, etiqueta) in enumerate(dataset.unbatch().take(n)):
        # Obtenemos el valor numérico y su forma (shape)
        valor = etiqueta.numpy()
        forma = etiqueta.shape
        tipo = etiqueta.dtype

        print(f"Muestra {i+1}: Valor = {valor} | Shape = {forma} | Tipo = {tipo}")

# 1. Verificamos el dataset de entrenamiento (el que tiene aumento)
inspeccionar_etiquetas(ds_train_process, "ds_train (Entrenamiento)")

# 2. Verificamos el dataset de prueba (el que causaba el error de ndim)
inspeccionar_etiquetas(ds_test_process, "ds_test (Prueba)")

In [ ]:
def one_hot_map(imagen, etiqueta):
    etiqueta = tf.one_hot(etiqueta, depth=num_classes)
    return imagen, etiqueta

ds_train_process = ds_train_process.map(one_hot_map)
ds_test_process  = ds_test_process.map(one_hot_map)

In [ ]:
# ==========================================
# 1. Configuración de Rutas y Callbacks V2
# ==========================================
from google.colab import drive
import os
import glob
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, regularizers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

drive.mount('/content/drive')

EPOCHS = 60

checkpoint_dir = '/content/drive/MyDrive/ModelosVGG/'
# Nombre solicitado: vgg_latest__v2.keras
checkpoint_path = os.path.join(checkpoint_dir, 'vgg_latest__v2.keras')

if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

# ==========================================
# 2. Construcción con Flatten (Requisito)
# ==========================================

model_files = glob.glob(checkpoint_path)

if model_files:
    print(f"Cargando modelo V2: {checkpoint_path}")
    model = tf.keras.models.load_model(checkpoint_path)
else:
    print("Construyendo arquitectura con Flatten y regularización reforzada...")

    vgg_base = VGG16(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # REGLA DE ORO: Al principio, congela TODO el VGG
    # El Flatten ya añade demasiada inestabilidad inicial
    vgg_base.trainable = False

    model = models.Sequential([
    vgg_base,
    layers.Flatten(),
    # Bajamos la complejidad: menos neuronas para que el Flatten no sea un monstruo
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
    ])


    model.compile(
    optimizer=optimizers.RMSprop(learning_rate=1e-3), # Subimos el ritmo
    loss='categorical_crossentropy',
    metrics=['accuracy'])


# ==========================================
# 3. Callbacks para el Guardado V2
# ==========================================

callbacks_list = [
    # Guarda solo si mejora val_accuracy (el criterio de verdad)
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    # Si el loss de validación sube durante 4 épocas, paramos (Anti-Overfitting)
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    # Si nos estancamos, bajamos el paso
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("\n--- Iniciando entrenamiento con Flatten y Regularización L2 ---")
history = model.fit(
    ds_train_process,
    validation_data=ds_test_process,
    epochs=EPOCHS,
    callbacks=callbacks_list
)

model.summary()

In [ ]:
# ==========================================
# 1. Configuración de Google Drive y Rutas
# ==========================================
from google.colab import drive
import os
import glob
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import ModelCheckpoint

# Montar Drive para persistencia
drive.mount('/content/drive')

# Parámetros
EPOCHS = 80
checkpoint_dir = '/content/drive/MyDrive/ModelosVGG/'
checkpoint_path = os.path.join(checkpoint_dir, 'vgg_latest.keras')

if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

# ==========================================
# 2. Construcción de la Arquitectura Estilo VGG16
# ==========================================

# Intentamos cargar modelo previo, si no, creamos uno nuevo
model_files = glob.glob(os.path.join(checkpoint_dir, '*.keras'))

if model_files:
    latest_model = max(model_files, key=os.path.getctime)
    print(f"Cargando modelo existente: {latest_model}")
    model = tf.keras.models.load_model(latest_model)
else:
    print("Construyendo arquitectura fiel a VGG16...")

    # Cargamos el extractor de características (los 5 bloques convolucionales)
    vgg_base = VGG16(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # Fine-Tuning: Congelamos los primeros 4 bloques y liberamos el último (Block 5)
    # Esto permite que los filtros de alto nivel se especialicen en rasgos de perros.
    vgg_base.trainable = True
    for layer in vgg_base.layers[:-4]:
        layer.trainable = False

    # Definición del modelo siguiendo el estilo de la "Cabeza" (Top) de VGG16
    model = models.Sequential([
        vgg_base,
        layers.Flatten(), # VGG16 original usa Flatten para conectar a las Dense

        # Capa Densa 1 (Inspirada en las 4096 originales, ajustada para eficiencia)
        layers.Dense(1024, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        # Capa Densa 2 (Inspirada en las 4096 originales)
        layers.Dense(1024, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        # Capa de salida: 120 neuronas para Stanford Dogs
        layers.Dense(num_classes, activation='softmax')
    ])

    # Compilación con Learning Rate bajo para no destruir los pesos de ImageNet
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

# ==========================================
# 3. Callbacks y Entrenamiento
# ==========================================

# Callback para guardar progreso en Drive
checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=False,
    monitor='val_accuracy',
    mode='max',
    save_best_only=False, # Guardamos cada época para retomar si se desconecta Colab
    verbose=1
)

print("\n--- Iniciando entrenamiento estilo VGG16 ---")
history = model.fit(
    ds_train_process,                         # Pipeline con Shuffle y Aumento
    validation_data=ds_test_process, # Pipeline de prueba "limpio"
    epochs=EPOCHS,
    callbacks=[checkpoint_callback]
)

# Mostrar la estructura final
model.summary()